## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

In [2]:
MODEL = "openai/gpt-4o-mini"
DB_NAME = "vector_db"
load_dotenv(override=True)

Out[1]: True


### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [4]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL)

### These LangChain objects implement the method `invoke()`

In [5]:
retriever.invoke("Who is Avery?")

Out[1]: 
[Document(id='312a18e4-7b84-4115-be7b-890ce7975495', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilie

In [6]:
llm.invoke("Who is Avery?")

---------------------------------------------------------------------------
APIStatusError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 llm.invoke("Who is Avery?")

File ~\projects\llm_engineering\.venv\Lib\site-packages\langchain_core\language_models\chat_models.py:382, in BaseChatModel.invoke(self, input, config, stop, **kwargs)
    368 @override
    369 def invoke(
    370     self,
   (...)    375     **kwargs: Any,
    376 ) -> AIMessage:
    377     config = ensure_config(config)
    378     return cast(
    379         "AIMessage",
    380         cast(
    381             "ChatGeneration",
--> 382             self.generate_prompt(
    383                 [self._convert_input(input)],
    384                 stop=stop,
    385                 callbacks=config.get("callbacks"),
    386                 tags=config.get("tags"),
    387                 metadata=config.get("metadata"),
    388                 run_name=config.get("run_name"),

## Time to put this together!

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

---------------------------------------------------------------------------
APIStatusError                            Traceback (most recent call last)
Cell In[1], line 1
----> 1 answer_question("Who is Averi Lancaster?", [])

Cell In[1], line 5, in answer_question(question, history)
      3 context = "\n\n".join(doc.page_content for doc in docs)
      4 system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
----> 5 response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
      6 return response.content

File ~\projects\llm_engineering\.venv\Lib\site-packages\langchain_core\language_models\chat_models.py:382, in BaseChatModel.invoke(self, input, config, stop, **kwargs)
    368 @override
    369 def invoke(
    370     self,
   (...)    375     **kwargs: Any,
    376 ) -> AIMessage:
    377     config = ensure_config(config)
    378     return cast(
    379         "AIMessage",
    380         cast(
    381             "ChatGeneration",
--> 3

## What could possibly come next? 😂

In [10]:
gr.ChatInterface(answer_question).launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
Out[1]: 


C:\Users\shobhasreenivas\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


## Admit it - you thought RAG would be more complicated than that!!